# 🧬 Cancer Prediction Using Machine Learning

## Phase 3 — Data Cleaning

**Objective:** Clean and prepare the raw dataset for analysis and modeling.

---

### 📋 Table of Contents

1. Setup & Load Raw Data
2. Drop Unnecessary Columns
3. Handle Missing Values (Verification)
4. Remove Duplicates (Verification)
5. Target Variable Encoding
6. Data Type Validation
7. Save Cleaned Dataset
8. Phase 3 Summary

---

### Why is Data Cleaning important?

**"Garbage in, garbage out"** — the most important rule in ML.

No matter how sophisticated your model is, if the data is dirty, the predictions
will be unreliable. In industry, data scientists spend **60-80% of their time**
on data cleaning and preparation.

### Cleaning vs. Feature Engineering

| Data Cleaning | Feature Engineering |
|---------------|--------------------|
| Fix what's wrong | Create what's useful |
| Remove noise | Add signal |
| Handle missing values | Create new features |
| Fix data types | Transform existing features |
| Done BEFORE EDA | Done AFTER EDA |

In [ ]:
# =============================================================================
# CELL 1: Setup & Imports
# =============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Add src/ to path
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from data_loader import load_raw_data, load_from_sklearn, save_raw_data
from utils import setup_logging, print_section_header

logger = setup_logging()
print("✅ Imports ready")

In [ ]:
# =============================================================================
# CELL 2: Load Raw Data
# =============================================================================
# WHY: We always start from the raw data — never from a previous
#       cleaning attempt. This ensures our pipeline is reproducible.
# =============================================================================

print_section_header("Loading Raw Data", "📦")

try:
    df = load_raw_data()
    print(f"✅ Loaded from data/raw/breast_cancer.csv")
except FileNotFoundError:
    print("⚠️ Raw data not found, loading from sklearn...")
    df = load_from_sklearn()
    save_raw_data(df)
    print(f"✅ Loaded from sklearn and saved to data/raw/")

print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns[:5])} ... ({len(df.columns)} total)")

# Keep a copy of original shape for comparison at the end
original_shape = df.shape
print(f"\n📋 Starting shape: {original_shape[0]} rows × {original_shape[1]} columns")

---

## 2. 🗑️ Drop Unnecessary Columns

### Which columns to drop and WHY:

| Column | Why Drop It? |
|--------|-------------|
| `id` | Patient identifier — has no predictive value. Including it would cause the model to memorize patient IDs instead of learning patterns. |
| `Unnamed: 32` | Some CSV versions have an extra empty column from trailing commas. |

### Why dropping `id` is critical

If we keep `id`, the model might learn that "patients with ID > 500 tend to be Malignant"
which is a **spurious correlation** — it doesn't generalize to new patients.

This is called **data leakage** through irrelevant features, and it's one of the most
common mistakes in ML projects.

In [ ]:
# =============================================================================
# CELL 3: Drop Unnecessary Columns
# =============================================================================
# WHY: 'id' is a patient identifier with no predictive value.
#       Some CSV versions also have an 'Unnamed: 32' column from
#       trailing commas in the source file.
#
# BEST PRACTICE: Use .drop() with errors='ignore' so the code
#       doesn't crash if the column doesn't exist.
# =============================================================================

print_section_header("Dropping Unnecessary Columns", "🗑️")

# Columns to drop
cols_to_drop = ['id', 'Unnamed: 32']

# Check which ones exist
existing_drops = [col for col in cols_to_drop if col in df.columns]
missing_drops = [col for col in cols_to_drop if col not in df.columns]

print(f"  Columns to drop: {cols_to_drop}")
print(f"  Found in dataset: {existing_drops}")
print(f"  Not found (skipped): {missing_drops}")

# Drop with safety check
df_cleaned = df.drop(columns=existing_drops, errors='ignore')

print(f"\n  Before: {df.shape[1]} columns")
print(f"  After:  {df_cleaned.shape[1]} columns")
print(f"  Dropped: {df.shape[1] - df_cleaned.shape[1]} column(s)")
print(f"\n✅ Remaining columns: {list(df_cleaned.columns[:5])} ... ({len(df_cleaned.columns)} total)")

### 💡 Why `errors='ignore'`?

Defensive programming! If someone runs this notebook with a different version
of the dataset that doesn't have `Unnamed: 32`, the code still works.

**Common beginner mistake:** Hard-coding column names without checking if they exist.
This causes `KeyError` crashes when the dataset changes.

**🎯 Interview Question:** *What is data leakage?*
> Data leakage occurs when information from outside the training dataset is used to create the model. Examples include: using future data to predict the past, including the target variable in the features, or using identifiers (like patient ID) that correlate with the target by chance.

---

## 3. ❓ Missing Values (Verification)

Even though Phase 2 showed no missing values, we verify again after dropping columns.
This is a **defensive coding practice** — assumptions should be validated at every step.

In [ ]:
# =============================================================================
# CELL 4: Verify No Missing Values After Column Drop
# =============================================================================

print_section_header("Missing Values Verification", "❓")

missing_total = df_cleaned.isnull().sum().sum()
print(f"  Total missing values: {missing_total}")

if missing_total == 0:
    print("  ✅ Confirmed: No missing values in cleaned dataset")
else:
    print("  ⚠️ Missing values found — handling...")
    # Show which columns have missing values
    missing_cols = df_cleaned.isnull().sum()
    print(missing_cols[missing_cols > 0])

---

## 4. 🔄 Duplicate Removal (Verification)

We verified in Phase 2 that there are no duplicates, but let's confirm after column drops.

In [ ]:
# =============================================================================
# CELL 5: Verify No Duplicates
# =============================================================================

print_section_header("Duplicate Check After Cleaning", "🔄")

dupes = df_cleaned.duplicated().sum()
print(f"  Duplicate rows: {dupes}")

if dupes == 0:
    print("  ✅ Confirmed: No duplicate rows")
else:
    print(f"  ⚠️ Found {dupes} duplicates — removing...")
    df_cleaned = df_cleaned.drop_duplicates()
    print(f"  ✅ Removed {dupes} duplicates. New shape: {df_cleaned.shape}")

---

## 5. 🏷️ Target Variable Encoding

### Why encode the target?

ML algorithms work with **numbers**, not strings. We need to convert:
- `M` (Malignant) → `1` (positive class — the class we want to detect)
- `B` (Benign) → `0` (negative class)

### Why M = 1?

By convention, the **positive class** (the one we want to detect) gets the value `1`.
Since our goal is to **detect cancer**, Malignant = 1.

This affects how we interpret metrics:
- **Recall** = TP / (TP + FN) → "Of all cancer patients, how many did we catch?"
- **Precision** = TP / (TP + FP) → "Of all patients we predicted as cancer, how many actually have it?"

### Encoding methods comparison:

| Method | When to use | Example |
|--------|------------|--------|
| **Label Encoding** | Binary or ordinal target | M→1, B→0 |
| **One-Hot Encoding** | Nominal features with >2 categories | color → [is_red, is_blue, is_green] |
| **Ordinal Encoding** | Ordinal features | size: S→0, M→1, L→2 |
| **Target Encoding** | High-cardinality categoricals | Replace category with mean target value |

For our binary target, **Label Encoding** is the correct choice.

In [ ]:
# =============================================================================
# CELL 6: Encode Target Variable
# =============================================================================
# WHY: ML models need numeric inputs. We encode:
#       M (Malignant) → 1 (positive class — cancer detected)
#       B (Benign)    → 0 (negative class — no cancer)
#
# METHOD: We use sklearn's LabelEncoder for consistency with the
#       ML ecosystem. We could also use .map({'M': 1, 'B': 0}),
#       but LabelEncoder is the industry standard.
#
# IMPORTANT: We're doing this BEFORE the train-test split because
#       encoding the target is a deterministic mapping, not a
#       learned transformation. There's no risk of data leakage.
# =============================================================================

from sklearn.preprocessing import LabelEncoder

print_section_header("Target Variable Encoding", "🏷️")

# Show before encoding
print("Before encoding:")
print(f"  Type: {df_cleaned['diagnosis'].dtype}")
print(f"  Values: {df_cleaned['diagnosis'].unique()}")
print(f"  Distribution: {df_cleaned['diagnosis'].value_counts().to_dict()}")

# Apply Label Encoding
le = LabelEncoder()
df_cleaned['diagnosis'] = le.fit_transform(df_cleaned['diagnosis'])

# Show after encoding
print("\nAfter encoding:")
print(f"  Type: {df_cleaned['diagnosis'].dtype}")
print(f"  Values: {df_cleaned['diagnosis'].unique()}")
print(f"  Distribution: {df_cleaned['diagnosis'].value_counts().to_dict()}")

# Show the mapping
print("\n  Mapping:")
for i, label in enumerate(le.classes_):
    emoji = '🟢' if label == 'B' else '🔴'
    print(f"    {emoji} {label} → {i}")

# IMPORTANT: Verify the mapping is correct
# LabelEncoder sorts alphabetically: B=0, M=1
assert le.transform(['B'])[0] == 0, "Benign should be 0"
assert le.transform(['M'])[0] == 1, "Malignant should be 1"
print("\n✅ Encoding verified: B=0 (Benign), M=1 (Malignant)")

### 💡 Why LabelEncoder and not `.map()`?

Both work for binary targets, but `LabelEncoder` has advantages:

1. **Consistency** — all sklearn tutorials use it
2. **Inverse transform** — `le.inverse_transform([1])` → `'M'` (useful for displaying predictions)
3. **Production ready** — can be serialized with the model pipeline

However, for multi-class features as input features (not targets), use `OrdinalEncoder` or `OneHotEncoder` instead.

**🎯 Interview Question:** *When would you use One-Hot Encoding vs Label Encoding?*
> Use Label Encoding for binary targets or ordinal features with a natural order (low/medium/high). Use One-Hot Encoding for nominal features with no natural order (red/blue/green) to avoid implying a false mathematical relationship between categories.

---

## 6. ✅ Data Type Validation

Final check to ensure every column has the correct data type.

In [ ]:
# =============================================================================
# CELL 7: Final Data Type Validation
# =============================================================================
# WHY: One final sanity check before saving. All features should be
#       float64, and the target should be int (0 or 1).
# =============================================================================

print_section_header("Final Data Type Validation", "✅")

dtype_summary = df_cleaned.dtypes.value_counts()
print("Data type distribution:")
for dtype, count in dtype_summary.items():
    print(f"  {str(dtype):10s} → {count} columns")

# Verify all features are numeric
non_numeric = df_cleaned.select_dtypes(exclude=[np.number]).columns
if len(non_numeric) == 0:
    print("\n✅ All columns are numeric — ready for ML!")
else:
    print(f"\n⚠️ Non-numeric columns remaining: {list(non_numeric)}")

# Final shape
print(f"\n📊 Final cleaned dataset:")
print(f"   Rows:     {df_cleaned.shape[0]}")
print(f"   Columns:  {df_cleaned.shape[1]}")
print(f"   Features: {df_cleaned.shape[1] - 1}")
print(f"   Target:   diagnosis (0=Benign, 1=Malignant)")

# Show the cleaned DataFrame
print("\n🔍 First 5 rows of cleaned data:")
df_cleaned.head()

---

## 7. 💾 Save Cleaned Dataset

We save the cleaned dataset to `data/processed/` so all subsequent notebooks
can load it directly without repeating the cleaning steps.

In [ ]:
# =============================================================================
# CELL 8: Save Cleaned Dataset
# =============================================================================
# WHY: Saving to data/processed/ means:
#       1. Subsequent notebooks don't repeat cleaning logic
#       2. We have a clear separation between raw and processed data
#       3. The cleaning step is reproducible from raw data
#
# BEST PRACTICE: Use index=False to avoid saving the DataFrame index
#       as an extra unnamed column.
# =============================================================================

print_section_header("Saving Cleaned Dataset", "💾")

# Save to processed directory
processed_path = os.path.join('..', 'data', 'processed', 'breast_cancer_cleaned.csv')
os.makedirs(os.path.dirname(processed_path), exist_ok=True)

df_cleaned.to_csv(processed_path, index=False)

# Verify the save
saved_df = pd.read_csv(processed_path)
assert saved_df.shape == df_cleaned.shape, "Shape mismatch after save!"
assert list(saved_df.columns) == list(df_cleaned.columns), "Column mismatch after save!"

print(f"  ✅ Saved to: {processed_path}")
print(f"  ✅ File size: {os.path.getsize(processed_path) / 1024:.1f} KB")
print(f"  ✅ Verification: Shape {saved_df.shape} matches original")
print(f"\n📊 Cleaning Summary:")
print(f"   Original:  {original_shape[0]} rows × {original_shape[1]} columns")
print(f"   Cleaned:   {df_cleaned.shape[0]} rows × {df_cleaned.shape[1]} columns")
print(f"   Rows removed: {original_shape[0] - df_cleaned.shape[0]}")
print(f"   Columns removed: {original_shape[1] - df_cleaned.shape[1]} (id)")

---

## 8. 📋 Phase 3 Summary

### ✔ Summary

In Phase 3, we cleaned the raw dataset:
- **Dropped** the `id` column (not predictive, potential leakage)
- **Verified** no missing values or duplicates after cleaning
- **Encoded** the target: M → 1 (Malignant), B → 0 (Benign) using LabelEncoder
- **Validated** all data types are numeric and ready for ML
- **Saved** the cleaned dataset to `data/processed/breast_cancer_cleaned.csv`

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| Data Leakage | Including irrelevant features that correlate with target |
| Label Encoding | Converting categorical target to numeric (B→0, M→1) |
| Defensive Coding | Checking assumptions at every step |
| Idempotent Pipeline | Cleaning that produces same result every run |
| Raw vs Processed | Never modify raw data — save cleaned version separately |

### ✔ Files Created

```
📁 data/processed/
   └── 📄 breast_cancer_cleaned.csv    ← Cleaned dataset (569 rows × 31 cols)
```

### ✔ Next Phase Preview: Phase 4 — Exploratory Data Analysis (EDA)

In the next phase, we will create 8+ visualizations:
1. Feature distribution histograms
2. Correlation heatmap (30×30 matrix)
3. Scatter plots of top features
4. Violin plots by diagnosis
5. Pairplot for top 5 features

---

*Phase 3 Complete ✅ — Proceeding to Phase 4*